In [1]:
import json
import os
import chromadb
from typing import Annotated, TYPE_CHECKING

from IPython.display import display, HTML

from openai import AsyncOpenAI

from semantic_kernel.agents import ChatCompletionAgent, ChatHistoryAgentThread
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.contents import FunctionCallContent,FunctionResultContent, StreamingTextContent, ChatMessageContent
from semantic_kernel.contents.utils.author_role import AuthorRole
from semantic_kernel.functions import kernel_function

from semantic_kernel.connectors.ai.open_ai import OpenAIChatPromptExecutionSettings
from semantic_kernel.contents.chat_history import ChatHistory
from semantic_kernel.contents import AuthorRole

if TYPE_CHECKING:
    from chromadb.api.models.Collection import Collection
# Initialize the asynchronous OpenAI client
from dotenv import load_dotenv

from typing import Annotated, Dict, Any, List
from semantic_kernel.functions import kernel_function

In [2]:
import json
import os

# ------------------------------
# 动态加载 subcommand 信息
# ------------------------------
def load_subcommands(info_path="../json/dorado_subcommand_info.json", param_path="../json/dorado_subcommand_parameter.json"):
    with open(info_path, "r") as f:
        subcommand_info = json.load(f)
    with open(param_path, "r") as f:
        subcommand_parameter = json.load(f)
    return subcommand_info, subcommand_parameter

# 使用示例
SUBCOMMANDINFO, subcommand_parameter = load_subcommands()

In [3]:
# ------------------------------
# 4️⃣ 加载 ChromaDB
# ------------------------------
chroma_client = chromadb.PersistentClient(path="../db/dorado_db")
collection = chroma_client.get_or_create_collection(
    name="dorado_documents",
    metadata={"description": "dorado_import_documents"}
)
# 插入示例文档
documents = [
    "Before basecalling, it's necessary to ask user whether he or she has downloaded the model. If not, tell him use subcommand 'download_model' of download to finish the task.",
    
]
collection.add(
    documents=documents,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[{"source": "training"} for _ in documents],
)

In [4]:
import json
from typing import Dict, Any

class CodeGeneratorPlugin:
    def __init__(self, collection: "Collection", subcommand_parameter_dict: Dict[str, Any] = None):
        self.subcommand_parameter = subcommand_parameter_dict
        self.collection = collection

    @kernel_function(
        description="Get the subcommand parameter",
        name="get_subcommand_parameter"
    )
    def get_subcommand_parameter(self, subcommand_name: str) -> Dict[str, Any]:
        return self.subcommand_parameter[subcommand_name]


    @kernel_function(
        description="Get the document context",
        name="get_document_context"
    )
    def get_document_context(self, user_query: str) -> str:
        return self.collection.query(query_texts=[user_query], n_results=10)


    @kernel_function(
        description="Generate a prompt for the LLM given tool parameters JSON and the user's task request.",
        name="generate_tool_prompt"
    )
    def generate_tool_prompt(self,tool_definition: Dict[str, Any], user_task: str) -> str:
        """
        Generate a prompt for the LLM given tool parameters JSON and the user's task request.
        English only.
        """
        formatted_params = json.dumps(tool_definition, indent=2, ensure_ascii=False)
        prompt = (
            "You are an expert systems engineer skilled in integrating command-line tools. "
            "You will be given a parameter definition and a user task. "
            "Your goal is to generate accurate codes that fulfills the user's request.\n\n"
            f"User Task:\n{user_task}\n\n"
            "Parameter Definition:\n"
            f"{formatted_params}\n\n"
            "Instructions:\n"
            "1. Use relevant optional parameters if they match the user's intent.\n"
            "2. Only use the parameters that are shown in the parameter definition. Never use parameters that are not in the parameter definition."
            "3. Do not use space as a parameter value. Use a name instead."
        )
        return prompt
    


In [5]:
# ------------------------------
# 1️⃣ 初始化 OpenAI 客户端
# ------------------------------
load_dotenv()
client = AsyncOpenAI(
    api_key=os.environ["GITHUB_TOKEN"],
    base_url="https://models.inference.ai.azure.com/"
)

chat_completion_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o",
    async_client=client,
)

In [6]:
from pydantic import BaseModel, ValidationError, Field

class SubTask(BaseModel):
    assigned_subcommand: str = Field(
        description="The specific subcommand assigned to handle this subtask")
    task_details: str = Field(
        description="Detailed description of what needs to be done for this subtask")


class PreparePlan(BaseModel):
    main_task: str = Field(
        description="The overall request from the user")
    subtasks: List[SubTask] = Field(
        description="List of subtasks broken down from the main task, each assigned to a specialized subcommand")

In [7]:
from semantic_kernel.functions import KernelArguments
AGENT_NAME = "Prepare_Agent"

AGENT_INSTRUCTIONS = """You are an planner agent.
    Your job is to decide which subcommand to run based on the user's request.
    Below are the available agents specialised in different tasks:
"""

for name, info in SUBCOMMANDINFO.items():
    AGENT_INSTRUCTIONS += ("\n - "+name+": "+info.get("description")+"\t"+ "Input: "+info.get("input"))


# Create the prompt execution settings and configure the Pydantic model response format
settings = OpenAIChatPromptExecutionSettings(response_format=PreparePlan)

pre_agent = ChatCompletionAgent(
    service=chat_completion_service,
    description="You are an planner agent.",
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
    arguments=KernelArguments(settings) 
)

In [8]:
from semantic_kernel.functions import KernelArguments
AGENT_NAME = "CodeGeneratorAgent"

AGENT_INSTRUCTIONS = """
    ### Role
    You are the Dorado Code Generator Agent, an expert in Oxford Nanopore data processing. You generate precise commands while providing intelligent, context-aware suggestions for downstream analysis.

    ### Workflow

    #### 1. Smart Information Audit (The Delta Check)
    Analyze input to identify **MISSING** variables. Acknowledge what is known; only ask for the unknown:
    * **Goal**: (basecall, demux, or alignment)
    * **Molecule**: (DNA or RNA)
    * **Modifications**: (Which specific mods, or "none")

    #### 2. Code Generation (Once all variables are known)
    - **Model Precision**: 
        - Single Mod: Use single-mod suffix (e.g., `_2OmeC@v1`).
        - Combined Mods: Use combined suffix ONLY if multiple mods are requested (e.g., `_m5C_2OmeC@v1`).
    - **Sync Flag**: `--modified-bases` MUST exactly match the modifications in the model name.
    - **Accuracy**: Default to `hac` unless `sup` is required for that specific mod combination.
    - **Syntax**: Wrap code in <code>...</code>. Use `[INPUT_PATH]` and `[OUTPUT_BAM]`.

    #### 3. Proactive Expert Suggestions (Post-Command)
    After providing the code, offer these optional value-adds:
    - **Visualization Tip**: "If you plan to plot signal traces or perform signal-to-base alignment later, I suggest adding `--emit-moves` to your command."
    - **Downstream Summary**: Ask if they want a summary: "Would you like me to generate a command to produce a `summary.txt` file for this run to check translocation speeds and Q-scores?"

    ---

    ### Interaction Protocol Examples

    **User:** "Basecall RNA for 2OmeC."
    **Agent:** "Got it. Preparing the command for RNA basecalling with 2OmeC detection.
    
    <code>dorado basecaller rna004_130bps_sup@v5.2.0_2OmeC@v1 [INPUT_PATH] --modified-bases 2OmeC > [OUTPUT_BAM]</code>

    **Expert Tips:**
    - **Plotting:** If you need to visualize signal traces later, consider adding `--emit-moves`.
    - **Next Step:** Would you like the command to generate a summary file for this run to evaluate sequencing quality?"

    **User:** "I want to basecall RNA."
    **Agent:** "I see you're working with RNA. Do you need to detect any specific modifications (e.g., m6A, m5C, 2OmeC), or should I proceed with standard basecalling?"
"""

code_agent = ChatCompletionAgent(
    service=chat_completion_service,
    description="You are an code generator agent.",
    name=AGENT_NAME,
    instructions=AGENT_INSTRUCTIONS,
    plugins=[CodeGeneratorPlugin(collection, subcommand_parameter)],
)

In [9]:
from semantic_kernel.agents import SequentialOrchestration, GroupChatOrchestration, RoundRobinGroupChatManager

from semantic_kernel.contents import ChatMessageContent

def agent_response_callback(message: ChatMessageContent) -> None:
    print(f"# {message.name}\n{message.content}")


In [10]:
from semantic_kernel.contents import ChatHistorySummarizationReducer

# Configure reduction parameters
REDUCER_TARGET_COUNT = 1  # Target number of messages to keep after reduction
REDUCER_THRESHOLD =  4 # Trigger reduction when message count exceeds this

history_reducer = ChatHistorySummarizationReducer(
    service=chat_completion_service,
    target_count=REDUCER_TARGET_COUNT,
    threshold_count=REDUCER_THRESHOLD,
)

chat = SequentialOrchestration(
    members=[pre_agent, code_agent],
    agent_response_callback=agent_response_callback,
)


In [11]:
from semantic_kernel.agents.runtime import InProcessRuntime
runtime = InProcessRuntime()
runtime.start()

# Prepare_Agent
{"main_task":"Basecalling RNA raw data while fetching appropriate model.","subtasks":[{"assigned_subcommand":"download","task_details":"Retrieve the basecalling model suitable for RNA data."},{"assigned_subcommand":"basecaller","task_details":"Perform basecalling on the RNA raw data using the downloaded model."}]}
# CodeGeneratorAgent
It seems you want to basecall RNA, but I need some clarification before generating the exact command:

- **Specific Modifications**: Are there any RNA modifications (e.g., m6A, m5C, 2OmeC) you aim to detect, or should I proceed assuming no modifications?
  
Let me know, and I’ll proceed with crafting an accurate command!
# Prepare_Agent
{"main_task":"Basecalling and model preparation for RNA data, detecting 2OmeC modifications, with input /path/pod5, output /path/bam/test.bam.","subtasks":[{"assigned_subcommand":"download","task_details":"Download and prepare necessary models for 2OmeC detection in RNA data, ensuring compatibility with dat

In [12]:
user_inputs = ["I want to basecall my RNA raw data but I do not have the basecall model", "My data is RNA, 004, and I want to detect 2OmeC, my input is /path/pod5, out is /path/bam/test.bam","yes"]

async def main():
    thread = ChatHistoryAgentThread(chat_history=history_reducer)
    for user_input in user_inputs:
        history_reducer.add_user_message(user_input)
        orchestration_result = await chat.invoke(
            task=history_reducer.messages,
            runtime=runtime,
        )
        value = await orchestration_result.get(timeout=100)
        print(f"***** Final Result *****\n{value}")
        history_reducer.add_assistant_message(value.content)

        if len(thread) > 4:
            await thread.reduce()
    await runtime.stop_when_idle()

await main()

***** Final Result *****
It seems you want to basecall RNA, but I need some clarification before generating the exact command:

- **Specific Modifications**: Are there any RNA modifications (e.g., m6A, m5C, 2OmeC) you aim to detect, or should I proceed assuming no modifications?
  
Let me know, and I’ll proceed with crafting an accurate command!
***** Final Result *****
Understood! You're looking to basecall RNA data with 2OmeC modifications. The input will come from `/path/pod5`, and the output will be written to `/path/bam/test.bam`.

Let me prepare the appropriate command for this workflow.

<code>
dorado basecaller rna004_130bps_sup@v5.2.0_2OmeC@v1 /path/pod5 --modified-bases 2OmeC > /path/bam/test.bam
</code>

**Expert Tips:**
1. **Visualization:** If you plan to examine the signal-to-base alignment or visualize the traces later, add the `--emit-moves` flag to your command.
2. **Next Step:** Would you be interested in a command to generate a `summary.txt` file for this run to chec